# POLAR OFFLINE RECORDER

## IMPORTS AND SETUPS

In [1]:
import yaml
from pathlib import Path
import platform
import asyncio
from bleak import BleakClient

In [2]:
# Config data are stored in config.yaml
configpath = Path("config.yaml")
with open (configpath, "r") as f:
        config = yaml.safe_load(f)

# BELT
'''
NOTE
Mac uses UUID, while Linux uses MAC address for the same task. So check the OS first.
Linux is called Linux. Windows is called Windows. Mac is called... wait for it... Darwin
'''
os = platform.system()
BELT = config["belt"]["uuid"] if os == "Darwin" else config["belt"]["mac_address"]
belt_human_readable = config["belt"]["name"]
# HEART RATE SERVICE (HRS)
HRS = config["belt"]["heart_rate_service"]

# POLAR MEASUREMENT DATA CONTROL (PMDC)
PMDC = config["belt"]["pmd_control"]

# POLAR MEASUREMENT DATA - DATA (PMDD)
PMDD = config["belt"]["pmd_data"]

## CONNECT

### Test available services

In [3]:
print("Connecting. This may take up to a minute.")

async with BleakClient(BELT) as client:
    print(f"Connected to {belt_human_readable}.")

    print("Services:")
    for service in client.services:
        print(f"\nSERVICE {service.uuid}")
        for char in service.characteristics:
            print(f"  CHAR {char.uuid} | props={char.properties}")

Connecting. This may take up to a minute.
Connected to Polar H10 21922C2C.
Services:

SERVICE 00001800-0000-1000-8000-00805f9b34fb
  CHAR 00002a00-0000-1000-8000-00805f9b34fb | props=['read']
  CHAR 00002a01-0000-1000-8000-00805f9b34fb | props=['read']
  CHAR 00002a04-0000-1000-8000-00805f9b34fb | props=['read']
  CHAR 00002aa6-0000-1000-8000-00805f9b34fb | props=['read']

SERVICE 00001801-0000-1000-8000-00805f9b34fb
  CHAR 00002a05-0000-1000-8000-00805f9b34fb | props=['indicate']

SERVICE 0000180d-0000-1000-8000-00805f9b34fb
  CHAR 00002a37-0000-1000-8000-00805f9b34fb | props=['notify']
  CHAR 00002a38-0000-1000-8000-00805f9b34fb | props=['read']

SERVICE 0000180a-0000-1000-8000-00805f9b34fb
  CHAR 00002a29-0000-1000-8000-00805f9b34fb | props=['read']
  CHAR 00002a24-0000-1000-8000-00805f9b34fb | props=['read']
  CHAR 00002a25-0000-1000-8000-00805f9b34fb | props=['read']
  CHAR 00002a27-0000-1000-8000-00805f9b34fb | props=['read']
  CHAR 00002a26-0000-1000-8000-00805f9b34fb | props=['

In [7]:
ACC_MEASUREMENT_TYPE = 0x02
ECG_MEASUREMENT_TYPE = 0x00

ACC_GET_SETTINGS = bytearray([0x01, ACC_MEASUREMENT_TYPE])
ECG_GET_SETTINGS = bytearray([0x01, ECG_MEASUREMENT_TYPE])
BATTERY_LEVEL = "00002a19-0000-1000-8000-00805f9b34fb"

In [8]:
def handle_pmd_control(sender, data):
    print("PMD CONTROL:", data.hex(" "))

In [9]:
async with BleakClient(BELT) as client:
    print(f"Connected to {belt_human_readable}")

    #Check battery level
    battery = await client.read_gatt_char(BATTERY_LEVEL)
    print("Battery:", battery[0], "%")

    try:
        await asyncio.wait_for(
            client.start_notify(PMDC, handle_pmd_control),
            timeout=5
        )
        print("PMDC indication subscription started")

    except asyncio.TimeoutError:
        print("TIMEOUT: start_notify on PMDC did not finish")

    print("Requesting ACC settings...")
    await client.write_gatt_char(PMDC, ACC_GET_SETTINGS, response=True)

    await asyncio.sleep(2)

    print("Requesting ECG settings...")
    await client.write_gatt_char(PMDC, ECG_GET_SETTINGS, response=True)
    print("ECG command written")

    await asyncio.sleep(2)

    await client.stop_notify(PMDC)
    print("Done")

Connected to Polar H10 21922C2C
Battery: 50 %


BleakGATTProtocolError: (5, 'GATT Protocol Error: Insufficient Authentication')

In [10]:
ACC_OFFLINE_START = bytearray([
    0x02, 0x82,              # start, offline ACC: (1 << 7) | 2 = 0x82
    0x00, 0x01, 0xC8, 0x00,  # sample rate 200
    0x01, 0x01, 0x10, 0x00,  # resolution 16
    0x02, 0x01, 0x08, 0x00   # range ±8 g
])

ECG_OFFLINE_START = bytearray([
    0x02, 0x80,              # start, offline ECG: (1 << 7) | 0 = 0x80
    0x00, 0x01, 0x82, 0x00,  # sample rate 130
    0x01, 0x01, 0x0E, 0x00   # resolution 14
])

ACC_OFFLINE_STOP = bytearray([0x03, 0x02])
ECG_OFFLINE_STOP = bytearray([0x03, 0x00])

HR_MEASUREMENT_TYPE = 0x08  # likely HR in Polar PMD/offline API
HR_OFFLINE_START = bytearray([
    0x02, 0x8E   # start command, offline HR
])

HR_STOP = bytearray([
    0x03, 0x0E   # stop HR
])

In [11]:
async with BleakClient(BELT) as client:

    print(f"Connected to {belt_human_readable}")

    await client.start_notify(PMDC, handle_pmd_control)

    print("Starting offline ACC...")
    await client.write_gatt_char(
        PMDC,
        ACC_OFFLINE_START,
        response=True
    )

    await asyncio.sleep(10)

    print("Stopping ACC...")
    await client.write_gatt_char(
        PMDC,
        ACC_OFFLINE_STOP,
        response=True
    )

    await asyncio.sleep(2)

    await client.stop_notify(PMDC)

    print("Done")

Connected to Polar H10 21922C2C


BleakGATTProtocolError: (5, 'GATT Protocol Error: Insufficient Authentication')

In [14]:
async with BleakClient(BELT) as client:
    print(f"Connected to {belt_human_readable}")

    await client.start_notify(PMDC, handle_pmd_control)
    print("PMDC indication subscription started")

    print("Starting offline HR...")
    await client.write_gatt_char(PMDC, HR_OFFLINE_START, response=True)

    await asyncio.sleep(20)

    print("Stopping offline HR...")
    await client.write_gatt_char(PMDC, HR_STOP, response=True)

    await asyncio.sleep(2)

    await client.stop_notify(PMDC)
    print("Done")

Connected to Polar H10 21922C2C
PMDC indication subscription started
Starting offline HR...
PMD CONTROL: f0 02 8e 02
Stopping offline HR...
PMD CONTROL: f0 03 0e 02
Done


In [15]:
async with BleakClient(BELT) as client:
    await client.start_notify(PMDC, handle_pmd_control)
    feat = await client.read_gatt_char(PMDC)  # PMD feature bytes
    print("PMD feature raw:", feat.hex(" "))
    b1 = feat[1] if len(feat) > 1 else 0
    b2 = feat[2] if len(feat) > 2 else 0
    print("ECG:", bool(b1 & 0x01), "ACC:", bool(b1 & 0x04))
    print("OFFLINE_RECORDING:", bool(b2 & 0x20), "OFFLINE_HR:", bool(b2 & 0x40))
    await client.stop_notify(PMDC)

PMD feature raw: 0f 05 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00
ECG: True ACC: True
OFFLINE_RECORDING: False OFFLINE_HR: False


In [10]:
async with BleakClient(BELT) as client:

    print(f"Connected to {belt_human_readable}")

    await client.start_notify(PMDC, handle_pmd_control)

    print("Starting offline ECG...")
    await client.write_gatt_char(
        PMDC,
        ECG_OFFLINE_START,
        response=True
    )

    await asyncio.sleep(10)

    print("Stopping ECG...")
    await client.write_gatt_char(
        PMDC,
        ECG_OFFLINE_STOP,
        response=True
    )

    await asyncio.sleep(2)

    await client.stop_notify(PMDC)

    print("Done")

Connected to Polar H10 21922C2C
Starting offline ECG...
PMD CONTROL: f0 02 80 02
Stopping ECG...
PMD CONTROL: f0 03 00 06 00
Done
